# Data Slayer 4.0 Main Notebook

Objective: Mengklasifikasikan setiap klip ujian berdurasi 1 detik (30 frame) menjadi normal (0) atau cheating (1) secara independen, hanya dari klip itu sendiri dan data train, dengan blend lima model dan F1-Macro sebagai metrik evaluasi. Model B adalah DINOv3 ViT-H+ yang 12 blok teratasnya di-fine-tune dengan augmentasi.

# Sinkronisasi versi modules dan libraries

In [ ]:
import sys
print(sys.version)
%pip install -q scikit-learn==1.8.0 numpy==2.4.4 pandas==2.3.3 matplotlib==3.10.9 torch==2.11.0

# Import modules dan libraries

In [ ]:
import itertools
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
DATA = Path("../data")
DINO_CACHE = Path("../cache/dinov3_hplus_224x400_step1_bf16")
POSE_CACHE = Path("../cache/pose_vitpose_all")
LEV_CACHE = Path("../cache/levjepa_large_224_30")
REF_CACHE = Path("../cache/kaggle_15fps")
B_CACHE = Path("../cache/kaggle_15fps_aug_full")
B_TAG = "k12_15fps_aug_full"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_FOLDS = 5
CLIP_LEN = 30
SIDE_DELAY = 0.5
C_LR = 0.003
CHEAT_ACTIONS = ["Looking_Under_the_Desk", "Reading_Hidden_Notes", "Repeated_Peeking", "Using_Smartwatch"]
NOSE, LEYE, REYE, LEAR, REAR, LSH, RSH, LEL, REL, LWR, RWR = range(11)
np.random.seed(SEED)

# Import datasets

In [ ]:
rows = []
for path in sorted(DATA.glob("train/*/*/*.MOV")):
    view, action = path.stem.split("_", 1)
    rows.append({
        "video": f"{path.parent.parent.name}/{path.parent.name}/{path.stem}",
        "label": int(path.parent.parent.name == "cheat"),
        "subject": int(path.parent.name.split("_")[1]),
        "view": view,
        "action": action,
    })
train = pd.DataFrame(rows)
sub = pd.read_csv(DATA / "sample_submission.csv")
eye = pd.read_csv("train_action_start_eye_all.csv").set_index("video")["start_eye"]
print(train.shape, sub.shape, eye.shape)
train.head()

# EDA

### Awal aksi di train
Detik saat subjek berhenti menulis dan mulai melakukan aksi cheating, dibaca manual dari video train kamera front. Kamera side merekam take yang sama dengan jeda sekitar 0,5 detik, jadi awal aksinya digeser 0,5 detik.

In [ ]:
train["start"] = [np.nan if not l else eye.get(v.replace("/side_", "/front_"), np.nan) + (SIDE_DELAY if vw == "side" else 0.0)
                  for v, l, vw in zip(train["video"], train["label"], train["view"])]
cheat = train[train["label"] == 1]
print(f"video cheating: {len(cheat)} | awal aksi terbaca: {cheat['start'].notna().sum()} | tidak jelas: {cheat['start'].isna().sum()}")
cheat.groupby("action")["start"].describe()[["count", "min", "50%", "max"]].round(2)

# Aggregate
Setiap video train dipotong menjadi klip 1 detik (frame 0-29, 30-59, dst), sama seperti bentuk klip test, supaya model belajar dari satuan yang sama dengan yang diprediksi.

### Label per klip
Klip video cheating berlabel 1 jika sebagian besar klip ada setelah awal aksi, klip sebelum itu berlabel 0 karena subjek masih menulis. Video cheating yang awal aksinya tidak terbaca tidak dipakai karena label per detiknya tidak diketahui.

In [ ]:
pose_meta = pd.read_csv(POSE_CACHE / "pose_train_s30_meta.csv")
pose_kp = np.load(POSE_CACHE / "pose_train_s30.npy")
pose_box = np.load(POSE_CACHE / "box_train_s30.npy")
pose_meta["video"] = [("cheat/" if l == 1 else "normal/") + v.replace(".MOV", "") for v, l in zip(pose_meta["video"], pose_meta["label"])]
clip_rows, dino_clips, pose_clips = [], [], []
for r in train.itertuples():
    if r.label == 1 and np.isnan(r.start):
        continue
    frames = np.load(DINO_CACHE / "train" / (r.video.replace("/", "__") + ".npy"))
    idx = np.where(pose_meta["video"] == r.video)[0]
    idx = idx[np.argsort(pose_meta["start"].values[idx])]
    for c in range(min(len(frames) // CLIP_LEN, len(idx))):
        dino_clips.append(frames[c * CLIP_LEN:(c + 1) * CLIP_LEN].astype(np.float32))
        pose_clips.append((pose_kp[idx[c]], pose_box[idx[c]]))
        clip_rows.append({"video": r.video, "subject": r.subject, "view": r.view, "action": r.action, "second": c,
                          "label": int(r.label == 1 and c + 0.5 >= r.start)})
clips = pd.DataFrame(clip_rows)
y = clips["label"].values
print("klip train:", len(clips), "| porsi cheating:", y.mean().round(3))
clips.groupby("second")["label"].mean().round(2).to_frame("porsi klip cheating").T

### Pembagian fold per video
Semua video dari satu subjek berada di fold yang sama, sehingga skor validasi mengukur kemampuan model pada orang yang belum pernah dilihat. Fold yang sama dipakai untuk melatih backbone yang di-fine-tune.

In [ ]:
videos = train[["video", "subject", "action"]].copy()
videos["fold"] = -1
splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold, (_, val_idx) in enumerate(splitter.split(videos, videos["action"], videos["subject"])):
    videos.loc[val_idx, "fold"] = fold
clips["fold"] = clips["video"].map(videos.set_index("video")["fold"])
clips.groupby("fold").agg(subjek=("subject", "nunique"), klip=("video", "count"), cheating=("label", "mean")).round(3)

# Feature Extraction
Semua fitur dan prediksi dihitung oleh script atau notebook Kaggle dan disimpan di cache, notebook ini hanya membacanya:
- scripts/extract_dino30.py: DINOv3 ViT-H+ (CLS + rata-rata patch, 2560 dimensi) untuk setiap frame, 30 frame per klip test dan semua frame video train.
- scripts/extract_pose_all.py: pose 17 keypoint (RT-DETR untuk box orang, ViTPose untuk keypoint) pada 30 frame setiap klip.
- scripts/extract_levjepa.py: LeVJEPA-1.5-Large (model video, frozen) pada 30 frame setiap klip, dibagi dua bagian bergantian (frame genap dan ganjil).
- kaggle_15fps_aug/kaggle_finetune_15fps_aug_full.ipynb (dijalankan di Kaggle): model B, ViT-H+ dengan 12 blok teratas di-fine-tune pada semua video train, 15 frame per detik dalam format test, 2 epoch, augmentasi hide + weather + hflip, weights akhir dirata-rata (EMA). Lalu scripts/kaggle_infer_test.py menghitung prediksi B pada 30 frame setiap klip test.
- kaggle_15fps_aug/kaggle_finetune_15fps_aug_folds1to4.ipynb dengan FOLDS = [0], AUG = "0", EPOCHS = 1: model fine-tune yang tidak melihat fold 0. Prediksi fold 0-nya dipakai sebagai acuan skala B saat blend, karena model B sendiri dilatih pada semua data dan tidak punya prediksi held-out.

### Fitur DINOv3 per frame
Fitur DINOv3 setiap frame klip test, dalam urutan sample_submission.

In [ ]:
test_dino = np.load(DINO_CACHE / "test.npy").astype(np.float32)
print(len(dino_clips), dino_clips[0].shape, test_dino.shape)

### Fitur pose per frame dan per klip
17 keypoint tubuh di dalam box orang diubah menjadi 19 sinyal gerakan kepala, tangan, dan badan untuk setiap frame. Untuk model logistic regression sinyal itu diringkas per klip dengan mean, std, dan rata-rata perubahan antar frame.

In [ ]:
def pose_series(kp, box):
    h = box[:, 3:4] + 1e-6
    x, y_, s = (kp[..., 0] - box[:, 0:1]) / h, (kp[..., 1] - box[:, 1:2]) / h, kp[..., 2]
    msx, msy = (x[:, LSH] + x[:, RSH]) / 2, (y_[:, LSH] + y_[:, RSH]) / 2
    sh_w = np.abs(x[:, LSH] - x[:, RSH]) + 1e-3
    d_l = np.hypot(x[:, LWR] - x[:, NOSE], y_[:, LWR] - y_[:, NOSE])
    d_r = np.hypot(x[:, RWR] - x[:, NOSE], y_[:, RWR] - y_[:, NOSE])
    return pd.DataFrame({
        "head_yaw": (x[:, NOSE] - (x[:, LEAR] + x[:, REAR]) / 2) / sh_w,
        "head_down": y_[:, NOSE] - msy,
        "eye_yaw": (x[:, NOSE] - (x[:, LEYE] + x[:, REYE]) / 2) / sh_w,
        "ear_vis": s[:, LEAR] - s[:, REAR],
        "wrist_gap": np.hypot(x[:, LWR] - x[:, RWR], y_[:, LWR] - y_[:, RWR]),
        "hand_face": np.minimum(d_l, d_r),
        "lwr_dx": x[:, LWR] - msx, "lwr_dy": y_[:, LWR] - msy,
        "rwr_dx": x[:, RWR] - msx, "rwr_dy": y_[:, RWR] - msy,
        "lel_dy": y_[:, LEL] - msy, "rel_dy": y_[:, REL] - msy,
        "wrist_score": np.minimum(s[:, LWR], s[:, RWR]),
        "lean_x": msx, "lean_y": msy, "sh_tilt": y_[:, LSH] - y_[:, RSH], "sh_w": sh_w,
        "head_speed": np.r_[0, np.hypot(np.diff(x[:, NOSE]), np.diff(y_[:, NOSE]))],
        "hand_speed": np.r_[0, (np.hypot(np.diff(x[:, LWR]), np.diff(y_[:, LWR])) + np.hypot(np.diff(x[:, RWR]), np.diff(y_[:, RWR]))) / 2],
    })


def pose_signals(kp, box):
    return np.nan_to_num(pose_series(kp, np.repeat(box[None], len(kp), 0)).values.astype(np.float32))


def pose_vector(kp, box):
    s = pose_series(kp, np.repeat(box[None], len(kp), 0))
    return np.nan_to_num(np.concatenate([s.mean(), s.std(), s.diff().abs().mean()]))


test_ids = pd.read_csv(POSE_CACHE / "pose_test_meta.csv")["id"]
row_of_id = {i: k for k, i in enumerate(test_ids)}
test_kp, test_box = np.load(POSE_CACHE / "pose_test.npy"), np.load(POSE_CACHE / "box_test.npy")
test_pose = [(test_kp[row_of_id[i]], test_box[row_of_id[i]]) for i in sub["id"]]

X_pose = np.stack([pose_vector(kp, box) for kp, box in pose_clips])
X_pose_test = np.stack([pose_vector(kp, box) for kp, box in test_pose])
pose_seq = np.stack([pose_signals(kp, box) for kp, box in pose_clips])
pose_seq_test = np.stack([pose_signals(kp, box) for kp, box in test_pose])
pose_mean, pose_std = pose_seq.reshape(-1, pose_seq.shape[-1]).mean(0), pose_seq.reshape(-1, pose_seq.shape[-1]).std(0) + 1e-6
normalize = lambda a: np.clip((a - pose_mean) / pose_std, -5, 5).astype(np.float32)
pose_seq, pose_seq_test = normalize(pose_seq), normalize(pose_seq_test)
print(X_pose.shape, X_pose_test.shape, pose_seq.shape, pose_seq_test.shape)

### Fitur LeVJEPA per klip
Untuk setiap klip: 30 vektor rata-rata patch (satu per frame) dan 2 token CLS (satu per bagian frame genap dan ganjil).

In [ ]:
lev_clips = []
for v, c in zip(clips["video"], clips["second"]):
    lev_clips.append(np.load(LEV_CACHE / "train" / (v.replace("/", "__") + ".npy"), mmap_mode="r")[c].copy())
lev_test = np.load(LEV_CACHE / "test.npy")
print(len(lev_clips), lev_clips[0].shape, lev_test.shape)

### Prediksi model B
Model B memakai kepala 8 aksinya sendiri, tanpa logistic regression. Probabilitas cheating sebuah frame adalah jumlah probabilitas empat aksi cheating, dan skor klip adalah rata-rata dari 30 frame klip itu. Untuk klip test dipakai model B yang dilatih pada semua video train. Acuan skalanya adalah prediksi held-out fold 0 dari model fine-tune yang tidak melihat fold 0 (dua frame per klip, frame ke-0 dan ke-15).

In [ ]:
b_test = np.exp(np.load(B_CACHE / f"{B_TAG}_test_logp_ema.npy")[:, :, :4]).sum(2).mean(1)
ref_meta = pd.read_csv(REF_CACHE / "k12_15fps_fold0_val_meta.csv")
ref_meta["second"] = ref_meta["frame"] // CLIP_LEN
ref_meta["p"] = np.exp(np.load(REF_CACHE / "k12_15fps_fold0_val_logprobs.npy")[:, :4]).sum(1)
b_ref = clips.merge(ref_meta.groupby(["video", "second"], as_index=False)["p"].mean(), on=["video", "second"], how="left")["p"].values
holdout = (clips["fold"] == 0).values & ~np.isnan(b_ref)
print(len(b_test), "klip test | klip acuan fold 0:", holdout.sum(), "| rata-rata skor B test:", b_test.mean().round(3))

# Classifier Models

### Logistic Regression pada LeVJEPA
Fitur sebuah klip diringkas menjadi mean (apa yang terlihat), std dan rata-rata selisih antar frame (seberapa banyak berubah), lalu digabung dengan ringkasan pose untuk satu Logistic Regression. Setiap fold dihitung out-of-fold, prediksi test adalah rata-rata dari 5 model fold.

In [ ]:
def summarize(f):
    return np.concatenate([f.mean(0), f.std(0), np.abs(np.diff(f, axis=0)).mean(0)])


def logistic_regression(features, c):
    oof, test = np.zeros(len(clips)), []
    for fold in range(N_FOLDS):
        X_all, X_test = features(fold)
        tr, va = (clips["fold"] != fold).values, (clips["fold"] == fold).values
        model = make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=3000)).fit(X_all[tr], y[tr])
        oof[va] = model.predict_proba(X_all[va])[:, 1]
        test.append(model.predict_proba(X_test)[:, 1])
    return oof, np.mean(test, 0)

In [ ]:
def lev_vector(a):
    f = a[:30].astype(np.float32)
    return np.concatenate([f.mean(0), a[30:].astype(np.float32).mean(0), f.std(0), np.abs(np.diff(f, axis=0)).mean(0)])


X_lev = np.hstack([np.stack([lev_vector(a) for a in lev_clips]), X_pose])
X_lev_test = np.hstack([np.stack([lev_vector(a) for a in lev_test]), X_pose_test])
oof_lev, test_lev = logistic_regression(lambda fold: (X_lev, X_lev_test), 0.001)
print(f"LeVJEPA + pose, F1-Macro per klip (threshold 0.5): {f1_score(y, oof_lev > 0.5, average='macro'):.4f}")

### Model sequence
Model kecil yang membaca 30 frame klip secara berurutan, bukan hanya ringkasannya. Setiap frame diubah menjadi 128 angka (PCA), dikurangi rata-rata klipnya sendiri supaya yang tersisa hanya perubahan antar frame, lalu digabung dengan 19 sinyal pose frame itu.
- BiGRU: membaca frame satu per satu ke depan dan ke belakang sambil mengingat yang sudah dilihat.
- Transformer: setiap frame membandingkan dirinya dengan semua frame lain sekaligus.

In [ ]:
def build_sequences(frames, frames_test, n_components=128):
    rng = np.random.default_rng(0)
    pick = zip(rng.integers(len(frames), size=60000), rng.integers(CLIP_LEN, size=60000))
    sample = np.stack([frames[c][f] for c, f in pick]).astype(np.float32)
    scaler = StandardScaler().fit(sample)
    pca = PCA(n_components, random_state=0, svd_solver="randomized").fit(scaler.transform(sample))

    def project(clip_frames):
        out = [pca.transform(scaler.transform(np.asarray(clip_frames[i:i + 500], dtype=np.float32).reshape(-1, sample.shape[1]))).reshape(-1, CLIP_LEN, n_components)
               for i in range(0, len(clip_frames), 500)]
        return np.concatenate(out).astype(np.float32)

    S, S_test = project(frames), project(frames_test)
    spread = S.reshape(-1, n_components).std(0, keepdims=True)
    S, S_test = S / spread, S_test / spread
    S, S_test = S - S.mean(1, keepdims=True), S_test - S_test.mean(1, keepdims=True)
    return np.concatenate([S, pose_seq], 2), np.concatenate([S_test, pose_seq_test], 2)


class GRUNet(nn.Module):
    def __init__(self, d, h=96):
        super().__init__()
        self.inp = nn.Sequential(nn.Dropout(0.2), nn.Linear(d, h), nn.GELU())
        self.rnn = nn.GRU(h, h, batch_first=True, bidirectional=True)
        self.out = nn.Sequential(nn.Dropout(0.3), nn.Linear(4 * h, 1))

    def forward(self, x):
        o, _ = self.rnn(self.inp(x))
        return self.out(torch.cat([o.mean(1), o.max(1).values], 1)).squeeze(1)


class TransformerNet(nn.Module):
    def __init__(self, d, h=64):
        super().__init__()
        self.inp = nn.Sequential(nn.Dropout(0.2), nn.Linear(d, h))
        self.pos = nn.Parameter(torch.zeros(1, CLIP_LEN, h))
        self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(h, 4, 2 * h, 0.2, batch_first=True, norm_first=True), 2)
        self.out = nn.Sequential(nn.Dropout(0.3), nn.Linear(2 * h, 1))

    def forward(self, x):
        o = self.enc(self.inp(x) + self.pos)
        return self.out(torch.cat([o.mean(1), o.max(1).values], 1)).squeeze(1)


def fit_predict(Net, X_tr, y_tr, X_preds, seed, epochs):
    torch.manual_seed(seed)
    model = Net(X_tr.shape[2]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), 1e-3, weight_decay=0.05)
    steps = epochs * ((len(X_tr) + 127) // 128)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps)
    X, Y = torch.tensor(X_tr, device=DEVICE), torch.tensor(y_tr, dtype=torch.float32, device=DEVICE)
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(X), device=DEVICE)
        for i in range(0, len(X), 128):
            b = perm[i:i + 128]
            loss = nn.functional.binary_cross_entropy_with_logits(model(X[b] + 0.1 * torch.randn_like(X[b])), Y[b])
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
    model.eval()
    with torch.no_grad():
        return [torch.sigmoid(model(torch.tensor(p, device=DEVICE))).cpu().numpy() for p in X_preds]


def sequence_model(Net, S, S_test, epochs=25, seeds=3):
    oof, test = np.zeros(len(clips)), []
    for fold in range(N_FOLDS):
        tr, va = (clips["fold"] != fold).values, (clips["fold"] == fold).values
        preds = [fit_predict(Net, S[tr], y[tr], [S[va], S_test], seed, epochs) for seed in range(seeds)]
        oof[va] = np.mean([p[0] for p in preds], 0)
        test.append(np.mean([p[1] for p in preds], 0))
    return oof, np.mean(test, 0)

### BiGRU dan Transformer pada DINOv3
Urutan 30 frame DINOv3 + pose, dilatih 5 fold dengan 3 seed per fold.

In [ ]:
S_dino, S_dino_test = build_sequences(dino_clips, test_dino)
oof_gru, test_gru = sequence_model(GRUNet, S_dino, S_dino_test)
oof_tf, test_tf = sequence_model(TransformerNet, S_dino, S_dino_test)
print(f"BiGRU DINOv3, F1-Macro per klip (threshold 0.5): {f1_score(y, oof_gru > 0.5, average='macro'):.4f}")
print(f"Transformer DINOv3, F1-Macro per klip (threshold 0.5): {f1_score(y, oof_tf > 0.5, average='macro'):.4f}")

### Transformer pada LeVJEPA
Model yang sama pada 30 vektor LeVJEPA per frame + pose.

In [ ]:
S_lev, S_lev_test = build_sequences([a[:CLIP_LEN] for a in lev_clips], lev_test[:, :CLIP_LEN])
oof_lev_tf, test_lev_tf = sequence_model(TransformerNet, S_lev, S_lev_test)
print(f"Transformer LeVJEPA, F1-Macro per klip (threshold 0.5): {f1_score(y, oof_lev_tf > 0.5, average='macro'):.4f}")

# Blend

### Rank dan bobot
Probabilitas tiap model diubah menjadi rank (posisinya di antara prediksi held-out train model itu sendiri) supaya skalanya sama, lalu dirata-rata dengan bobot. Model B hanya punya prediksi held-out untuk fold 0, jadi acuan rank semua model adalah klip fold 0 itu. Bobot tetap B 2 dan keempat model lain 1, hasil pencarian bobot out-of-fold semua fold pada versi sebelumnya (pencarian ulang dengan semua model memberi hasil yang sama dalam derau). Threshold dipilih dari F1-Macro klip acuan yang tertinggi, tanpa melihat data test.

In [ ]:
members = {
    "B fine-tuned": (b_ref[holdout], b_test),
    "BiGRU DINOv3": (oof_gru[holdout], test_gru),
    "Transformer DINOv3": (oof_tf[holdout], test_tf),
    "LeVJEPA + pose": (oof_lev[holdout], test_lev),
    "Transformer LeVJEPA": (oof_lev_tf[holdout], test_lev_tf),
}
names = list(members)
WEIGHTS = (2, 1, 1, 1, 1)
y_ref = y[holdout]
rank = lambda p, ref: np.searchsorted(np.sort(ref), p) / len(ref)
oof_rank = np.stack([rank(members[n][0], members[n][0]) for n in names])
test_rank = np.stack([rank(members[n][1], members[n][0]) for n in names])
thresholds = np.round(np.arange(0.3, 0.96, 0.025), 3)
f1s = [f1_score(y_ref, np.average(oof_rank, axis=0, weights=WEIGHTS) > t, average="macro") for t in thresholds]
THRESHOLD = thresholds[int(np.argmax(f1s))]
print("urutan model:", names)
print(f"bobot {WEIGHTS} | threshold {THRESHOLD} | F1-Macro klip acuan {max(f1s):.4f}")

### Skor tiap model
F1-Macro klip acuan (fold 0) tiap model sendiri pada threshold terbaiknya, dibandingkan dengan blend.

In [ ]:
alone = {n: max(f1_score(y_ref, oof_rank[i] > t, average="macro") for t in thresholds) for i, n in enumerate(names)}
alone["blend"] = max(f1s)
pd.Series(alone, name="F1-Macro per klip").round(4).to_frame().T

### Threshold
F1-Macro klip acuan blend pada berbagai threshold, kurvanya datar di sekitar threshold terpilih.

In [ ]:
blend_oof = np.average(oof_rank, axis=0, weights=WEIGHTS)
curve = pd.Series({t: f1_score(y_ref, blend_oof > t, average="macro") for t in thresholds}, name="F1-Macro per klip")
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(curve.index, curve.values, marker="o")
ax.axvline(THRESHOLD, color="gray", linestyle="--")
ax.set_xlabel("threshold rank blend")
ax.set_ylabel("F1-Macro per klip")
plt.show()

# Submission
Setiap klip test diberi skor dari model-model di atas, hanya dari 30 frame klip itu sendiri, lalu rank-nya dibandingkan dengan prediksi held-out train. Tidak ada informasi dari klip test lain.

In [ ]:
blend_test = np.average(test_rank, axis=0, weights=WEIGHTS)
Path("submissions").mkdir(exist_ok=True)
submission = pd.DataFrame({"id": sub["id"], "label": (blend_test > THRESHOLD).astype(int)})
submission.to_csv("submissions/main.csv", index=False)
print("porsi klip cheating:", submission["label"].mean().round(3))
submission.head()